# 05 – Directly mapped types and final output

Reads the output of **04** (`nisjemodel-remote-sensing_norge_2026_25833.geo.parquet`)
and overrides the classification where directly mapped areas exist:

1. **NA-MC08 – Marint bunnsystem preget av oksygenmangel**
   from the MDir water type classification `NyTypologi2022`, types **1a, 2a, 3a**
   (`… med oksygenfattig bunnvann`, see `mnk.rules.WATER_TYPE_NAMES`).
   Only bottom with `nkm_lys` afotisk or eufotisk and `nkm_lm-tv = LM-TV_0` (permanently submerged, not intertidal).
2. **NA-MJ01 – Taretrålingsbunn**
   from NIVA kelp trawling areas 2020–2025. Applies to all bottom inside the areas.

The rules are defined in `mnk.rules.DIRECT_RULES`. Layers are applied in order; a later layer overrides an earlier one where they overlap.
Only affected polygons (spatial index) go through geometry operations.

The ungridded result is saved as `marint-naturkart-ungridded_norge_2026_25833.geo.parquet`
(EPSG:25833, source for PMTiles). Finally everything is split along the EEA 1 km grid
(`rute_fkid`) in EPSG:3035 and written as `marint-naturkart_norge_2026_3035`
(parquet + gpkg).

In [ ]:
import gc
import json
import tempfile
from pathlib import Path

import numpy as np
import geopandas as gpd
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import shapely
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

import mnk
import mnk.rules
import mnk.attributes
import mnk.utils

CRS = "EPSG:25833"
REPO = Path.cwd()

IN_PARQUET = REPO / f"{mnk.utils.to_filename('nisjemodel-remote-sensing', 'norge', '2026', '25833')}.geo.parquet"
# EEA reference grid (1 km), native EPSG:3035 LAEA. The grid split (§5) runs in 3035,
# so the gridded product is in 3035. rute_fkid is the EEA CellCode (e.g. "1kmE4052N4303").
GRID_URL = "gs://niva-geodata/MarintNaturKart/aux/NO_1km.geo.parquet"

for rule in mnk.rules.DIRECT_RULES:
    print(rule["nin_kode_fkid"], mnk.rules.NIN_NAMES[rule["nin_kode_fkid"]], rule["url"])

## 1. Load data

In [ ]:
final = gpd.read_parquet(IN_PARQUET)
assert final.crs.to_epsg() == 25833
print(f"Input (04): {len(final):,} polygons")

# datamodell k_sannsynlighet_klasse code (Lav=1, Middel=2, Høy=3); mapped here so a
# label change only needs a rerun of 05.
SANNSYNLIGHET_KODE = {"lav": 1, "medium": 2, "høy": 3}
_vals = set(final["sannsynlighet_beskrivelse"].dropna().unique())
assert _vals <= set(SANNSYNLIGHET_KODE), _vals
final["sannsynlighet_beskrivelse"] = final["sannsynlighet_beskrivelse"].map(SANNSYNLIGHET_KODE).astype("Int64")


def load_mask(rule):
    """Read the direct dataset of *rule* and return clean single polygons."""
    url = rule["url"]
    gdf = gpd.read_parquet(url) if url.endswith(".parquet") else gpd.read_file(url)
    if rule["column"] is not None:
        gdf = gdf.loc[gdf[rule["column"]].isin(rule["values"])]
    gdf = gdf[["geometry"]].to_crs(CRS)
    gdf = gdf.set_geometry(shapely.make_valid(gdf.geometry.to_numpy()), crs=CRS)
    gdf = gdf.explode(index_parts=False)
    gdf = gdf[gdf.geom_type == "Polygon"].reset_index(drop=True)
    print(f"{rule['nin_kode_fkid']}: {len(gdf):,} polygons ({gdf.area.sum() / 1e6:,.1f} km²)")
    return gdf

## 2. Generic override

`override` splits affected polygons into the part inside the mask (new type) and
the remainder (unchanged), same pattern as in 04.

In [3]:
def override(final, mask, code, method, source, candidates=None):
    """Assign NiN type *code* to the parts of *final* inside *mask*."""
    if candidates is None:
        candidates = pd.Series(True, index=final.index)
    sub = final.loc[candidates]
    hit = sub.sindex.query(mask.geometry, predicate="intersects")
    hit_idx = sub.index[np.unique(hit[1])]

    affected = final.loc[hit_idx].copy()
    unaffected = final.drop(index=hit_idx)
    affected["_src"] = np.arange(len(affected))

    pairs = gpd.sjoin(affected, mask[["geometry"]], how="inner", predicate="intersects")
    mask_geom = mask.geometry.to_numpy()[pairs["index_right"].to_numpy()]
    inter = shapely.intersection(pairs.geometry.to_numpy(), mask_geom)
    inside = pairs.set_geometry(inter, crs=CRS).drop(columns="index_right")
    inside = inside.explode(index_parts=False)
    inside = inside[(inside.geom_type == "Polygon") & ~inside.geometry.is_empty]
    # Overlapping mask polygons give duplicates – dissolve per source polygon.
    inside = inside.dissolve(by="_src", as_index=False).explode(index_parts=False)
    inside = inside[inside.geom_type == "Polygon"]

    clip = inside.dissolve(by="_src").geometry.reindex(affected["_src"].to_numpy())
    src = affected.geometry.to_numpy()
    has = clip.notna().to_numpy()
    rest = src.copy()
    rest[has] = shapely.difference(src[has], clip[has].to_numpy())
    rest = affected.assign(geometry=rest).explode(index_parts=False)
    rest = rest[(rest.geom_type == "Polygon") & ~rest.geometry.is_empty]

    inside["nin_kode_fkid"] = code
    inside["NiN_name"] = mnk.rules.NIN_NAMES[code]
    inside["nkm_prosess"] = mnk.attributes.NKM_PROSESS[code]
    inside["metode_klassifisering"] = method
    # Directly mapped -> this dataset is the authoritative source for the piece.
    inside["datakilde"] = source

    cols = [c for c in unaffected.columns if not c.startswith("_")]
    out = gpd.GeoDataFrame(
        pd.concat([unaffected[cols], rest[cols], inside[cols]], ignore_index=True), crs=CRS
    )
    print(f"{code}: affected {len(affected):,} -> {len(inside):,} pieces, "
          f"{inside.area.sum() / 1e6:,.1f} km²")
    return out

## 3. Apply direct rules in order

Each rule in `mnk.rules.DIRECT_RULES` is burned in; a later rule overrides an
earlier one where they overlap.

In [4]:
for rule in mnk.rules.DIRECT_RULES:
    mask = load_mask(rule)
    candidates = None
    if rule["where"]:
        candidates = np.logical_and.reduce(
            [final[col].isin(vals) for col, vals in rule["where"].items()]
        )
        candidates = pd.Series(candidates, index=final.index)
    final = override(
        final, mask, rule["nin_kode_fkid"],
        method=rule["metode_klassifisering"],
        source=rule["datakilde"],
        candidates=candidates,
    )
    del mask, candidates
    gc.collect()

print(f"Combined: {len(final):,} polygons")
print(final["nin_kode_fkid"].value_counts())

NA-MC08: 106 polygons (181.5 km²)


NA-MC08: affected 3,046 -> 4,708 pieces, 152.8 km²


NA-MJ01: 3,846 polygons (207.8 km²)


NA-MJ01: affected 7,076 -> 14,660 pieces, 208.5 km²
Combined: 5,119,525 polygons
nin_kode_fkid
NA-MA01    1683947
NA-MA04     552835
NA-MA02     407811
NA-MA05     157864
NA-MA03     154238
NA-MA06      73502
NA-MB02      18545
NA-MC01      17851
NA-MJ01      14660
NA-MC04      13428
NA-MC02       5248
NA-MC08       4708
NA-MF01         14
Name: count, dtype: int64


## 4. Assign nk_fid and save ungridded output

`nk_fid` is assigned per **contiguous same-`nin_kode_fkid` region** (connected
components of adjacent polygons), so every row of one nature-type polygon shares the
same `nk_fid` while keeping its finer attributes (sannsynlighet, axes, datakilde,
SeaBee class). Computed by labelling (STRtree adjacency + `scipy` connected
components) — no geometric dissolve.

The result is saved before the grid split as
`marint-naturkart-ungridded_norge_2026_25833.geo.parquet` (EPSG:25833, source for the
PMTiles, `scripts/pyramids.sh`).

In [ ]:
# nk_fid: one id per contiguous same-nin_kode_fkid region (connected component of
# adjacent polygons). Rows keep their finer attributes and share the region's nk_fid.
final = final.explode(index_parts=False).reset_index(drop=True)
geoms = final.geometry.values

pairs = shapely.STRtree(geoms).query(geoms, predicate="intersects")
m = pairs[0] < pairs[1]
left, right = pairs[0][m], pairs[1][m]
code = pd.factorize(final["nin_kode_fkid"])[0]
same = code[left] == code[right]
left, right = left[same], right[same]

n = len(final)
graph = coo_matrix((np.ones(left.size, dtype=bool), (left, right)), shape=(n, n))
_, labels = connected_components(graph, directed=False)

# number components by Hilbert of their members so nk_fid is spatially coherent
order = pd.Series(final.geometry.hilbert_distance().to_numpy()).groupby(labels).min().sort_values().index
rank = pd.Series(np.arange(1, order.size + 1), index=order)
final["nk_fid"] = ("NKM-" + pd.Series(rank.loc[labels].to_numpy()).astype(str).str.zfill(10)).to_numpy()
print(f"nk_fid: {order.size:,} contiguous polygons from {n:,} rows")

ungridded_fname = mnk.utils.to_filename("marint-naturkart-ungridded", "norge", "2026", CRS.split(":")[1])
final.to_parquet(REPO / f"{ungridded_fname}.geo.parquet", compression="snappy")
print(f"Saved {ungridded_fname}.geo.parquet: {len(final):,} polygons")

## 5. EEA 1 km grid setup

Polygons are split along the EEA 1 km reference grid (`NO_1km`) in its native
**EPSG:3035 (LAEA)**, so cells stay exact 1 km squares and the gridded output is in
3035 (the product is reprojected into it). Each fragment is tagged with `rute_fkid`
(the EEA `CellCode`) and `rute_grense` — `True` if the fragment was cut by a cell
boundary, `False` if the polygon lies entirely within one cell.

In [ ]:
CHUNK = 50_000

# Split in the EEA grid's native CRS (EPSG:3035 LAEA) so cells stay exact 1 km
# squares; the product is reprojected into it, so the gridded output is in 3035.
# rute_fkid is the EEA CellCode (e.g. "1kmE4052N4303").
GRID_CRS = "EPSG:3035"
grid = gpd.read_parquet(GRID_URL, columns=["CellCode", "geometry"]).rename(
    columns={"CellCode": "rute_fkid"}
)
assert grid.crs.to_epsg() == 3035
final = final.to_crs(GRID_CRS)

GRID_BOUNDS = tuple(final.total_bounds)  # fixed bounds -> globally comparable Hilbert
minx, miny, maxx, maxy = GRID_BOUNDS
grid = grid.cx[minx:maxx, miny:maxy].reset_index(drop=True)
grid_geom = grid.geometry.to_numpy()
grid_id = grid["rute_fkid"].to_numpy()
grid_sindex = grid.sindex
print(f"Grid cells in extent: {len(grid):,}")


def split_chunk(chunk):
    """Split *chunk* by the grid; fragments carry rute_fkid and rute_grense."""
    src_idx, cell_idx = grid_sindex.query(chunk.geometry, predicate="intersects")
    geoms = chunk.geometry.to_numpy()[src_idx]
    cells = grid_geom[cell_idx]
    # Polygons fully inside one cell are not cut; the rest get a grid-cell edge.
    inside = shapely.contains_properly(cells, geoms) | shapely.covers(cells, geoms)
    out_geom = geoms.copy()
    out_geom[~inside] = shapely.intersection(geoms[~inside], cells[~inside])
    out = chunk.iloc[src_idx].drop(columns="geometry").reset_index(drop=True)
    out["rute_fkid"] = grid_id[cell_idx]
    out["rute_grense"] = ~inside
    out = gpd.GeoDataFrame(out, geometry=out_geom, crs=GRID_CRS)
    out = out.explode(index_parts=False)
    out = out[(out.geom_type == "Polygon") & ~out.geometry.is_empty]
    return out.reset_index(drop=True)

## 6. Split, assign fragment ids, sort, and save

Each polygon is split by the grid; every fragment gets a unique `nk_fragment_id`
(`<nk_fid>-NNNN`) while keeping the shared `nk_fid` from §4, so fragments stay linked
to their parent polygon. The output is sorted by `rute_fkid`, then Hilbert distance
within each cell, for downstream performance.

**Memory**: the split runs in chunks to parquet parts; the parts are concatenated and
sorted at the pyarrow level (compact WKB geometry, not shapely objects), the GeoParquet
is written, and the GPKG is written back from it in chunks.

In [ ]:
fname = mnk.utils.to_filename("marint-naturkart", "norge", "2026", GRID_CRS.split(":")[1])
parquet_path = REPO / f"{fname}.geo.parquet"
gpkg_path = REPO / f"{fname}.gpkg"
gpkg_path.unlink(missing_ok=True)

OUT_COLS = [
    "nk_fid", "nk_fragment_id", "opphav_kode", "rute_fkid", "rute_grense",
    "nin_kode_fkid", "NiN_name", "nkm_prosess",
    mnk.rules.COL_LYS, mnk.rules.COL_SALINITET, mnk.rules.COL_SUBSTRAT, mnk.rules.COL_TOERFALL,
    "metode_klassifisering", "sannsynlighet_beskrivelse", "datakilde", "nkm_seabee_klasse",
]

with tempfile.TemporaryDirectory(dir=REPO) as tmp:
    parts = []
    n_frag = 0
    for start in range(0, len(final), CHUNK):
        out = split_chunk(final.iloc[start : start + CHUNK])
        # fixed GRID_BOUNDS -> Hilbert comparable across parts for the global sort
        out["_hilbert"] = out.geometry.hilbert_distance(total_bounds=GRID_BOUNDS)
        part = Path(tmp) / f"part_{start:010d}.parquet"
        out.to_parquet(part, compression="snappy")
        parts.append(part)
        n_frag += len(out)
        print(f"  split {min(start + CHUNK, len(final)):,}/{len(final):,} -> {n_frag:,} fragments")
        del out
        gc.collect()
    del final
    gc.collect()

    # sort at the pyarrow level (WKB geometry) to keep peak memory bounded
    geo = json.loads(pq.read_schema(parts[0]).metadata[b"geo"])
    table = pa.concat_tables(pq.read_table(p) for p in parts)

for col in geo["columns"].values():
    col.pop("bbox", None)

table = table.sort_by([("rute_fkid", "ascending"), ("_hilbert", "ascending")])
table = table.drop_columns(["_hilbert"])

nk_fid = table.column("nk_fid").to_pandas()
frag_no = (nk_fid.groupby(nk_fid).cumcount() + 1).map("{:04d}".format)
table = table.append_column("nk_fragment_id", pa.array(nk_fid.str.cat(frag_no, sep="-"), type=pa.string()))

table = table.select(OUT_COLS + ["geometry"]).replace_schema_metadata(
    {b"geo": json.dumps(geo).encode()}
)
pq.write_table(table, parquet_path, compression="snappy")
print(f"Saved {table.num_rows:,} fragments -> {parquet_path.name}")

written = 0
first = True
for batch in pq.ParquetFile(parquet_path).iter_batches(batch_size=200_000):
    tb = pa.Table.from_batches([batch])
    geom = shapely.from_wkb(tb.column("geometry").to_pandas().to_numpy())
    g = gpd.GeoDataFrame(tb.drop_columns(["geometry"]).to_pandas(), geometry=geom, crs=GRID_CRS)
    g.to_file(gpkg_path, driver="GPKG", layer="nin", mode="w" if first else "a")
    first = False
    written += len(g)
    print(f"  gpkg {written:,}/{table.num_rows:,}")
print(f"Saved {written:,} fragments -> {gpkg_path.name}")

del table
gc.collect()